# Setup and load study boundary

In [ ]:
# %pip install "h3>=4.2,<5"

from pathlib import Path
import json
import shutil

import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
import shapely
from IPython.display import display
from pyproj import Transformer
from shapely.geometry import Polygon, box, mapping

PROJECT_ROOT = Path.home() / "padova_data"
BASE = PROJECT_ROOT / "gnn_outputs"
ZONES_FILE = PROJECT_ROOT / "Densità della popolazione/SIT_SEZIONI_2021/Sez_censimento_2021_32632_fixed.gpkg"

OUT_RES = BASE / "h3_padova_resolution_v1"
OUT_RES.mkdir(parents=True, exist_ok=True)

# Load census sections and dissolve into one study boundary
zones = gpd.read_file(ZONES_FILE).to_crs("EPSG:32632")
study = shapely.union_all(shapely.make_valid(zones.geometry.array))
study_area_m2 = study.area

print(f"Loaded {len(zones)} census sections, total area: {study_area_m2 / 1e6:.2f} km^2")

## Compare H3 resolutions (10 vs 11) and clip hexagons

In [ ]:
TARGET_AREA_M2 = 90.0 * 90.0
RESOLUTIONS = [10, 11]
CRS_M = "EPSG:32632"

# Expand bounding box by 300 m so boundary hexagons are not missed
xmin, ymin, xmax, ymax = study.bounds
search_box = box(xmin - 300, ymin - 300, xmax + 300, ymax + 300)
search_wgs84 = gpd.GeoSeries([search_box], crs=CRS_M).to_crs(4326).iloc[0]
search_shape = h3.geo_to_h3shape(mapping(search_wgs84))

def build_candidate_grid(res):
    ids = sorted(h3.h3shape_to_cells_experimental(search_shape, res, contain="overlap"))
    polygons = [Polygon([(lon, lat) for lat, lon in h3.cell_to_boundary(c)]) for c in ids]

    full = gpd.GeoDataFrame({"h3_id": ids, "resolution": res}, geometry=polygons, crs="EPSG:4326").to_crs(CRS_M)
    full = full[full.geometry.intersects(study)].copy()
    full["full_area_m2"] = full.geometry.area

    # Clip border hexagons to the study boundary
    inside = full.geometry.within(study)
    parts = full.geometry.copy()
    parts.loc[~inside] = full.loc[~inside].geometry.intersection(study)

    full["inside_area_m2"] = parts.area
    full = full[full["inside_area_m2"] > 0].copy()
    parts = parts.loc[full.index]
    full["inside_fraction"] = full["inside_area_m2"] / full["full_area_m2"]

    # Project cell centers to EPSG:32632
    latlon = np.array([h3.cell_to_latlng(c) for c in full["h3_id"]])
    centers = gpd.GeoSeries(gpd.points_from_xy(latlon[:, 1], latlon[:, 0]), crs="EPSG:4326").to_crs(CRS_M)
    full["center_x"] = centers.x.to_numpy()
    full["center_y"] = centers.y.to_numpy()

    # Compute distances between adjacent hexagons
    coords = dict(zip(full["h3_id"], zip(full["center_x"], full["center_y"])))
    distances = []
    for cell, (x, y) in coords.items():
        for other in h3.grid_disk(cell, 1):
            if other in coords and cell < other:
                ox, oy = coords[other]
                distances.append(np.hypot(ox - x, oy - y))
    distances = np.array(distances)

    median_area = full["full_area_m2"].median()
    coverage = full["inside_area_m2"].sum() / study_area_m2

    stats = {
        "resolution": res,
        "candidate_cells": len(full),
        "median_full_area_m2": median_area,
        "area_ratio_to_90m_square": median_area / TARGET_AREA_M2,
        "median_neighbor_distance_m": np.median(distances),
        "p05_neighbor_distance_m": np.percentile(distances, 5),
        "p95_neighbor_distance_m": np.percentile(distances, 95),
        "study_area_covered_pct": 100 * coverage,
        "abs_log_area_ratio": abs(np.log(median_area / TARGET_AREA_M2)),
    }

    clipped = full.copy()
    clipped.geometry = parts
    return full.reset_index(drop=True), clipped.reset_index(drop=True), stats

grids = {}
rows = []
for res in RESOLUTIONS:
    full, clipped, stats = build_candidate_grid(res)
    grids[res] = (full, clipped)
    rows.append(stats)

comparison = pd.DataFrame(rows).sort_values("resolution")
selected = int(comparison.loc[comparison["abs_log_area_ratio"].idxmin(), "resolution"])
comparison["selected"] = comparison["resolution"] == selected
comparison.to_csv(OUT_RES / "resolution_comparison.csv", index=False)
display(comparison.round(4))

# Save selected resolution layers
selected_full, selected_part = grids[selected]
selected_full.to_file(OUT_RES / f"h3_r{selected}_full_cells.gpkg", layer="full_cells", driver="GPKG", index=False)
selected_part.to_file(OUT_RES / f"h3_r{selected}_study_part.gpkg", layer="study_part", driver="GPKG", index=False)

# Plot full city and zoomed comparison
fig, axes = plt.subplots(1, 3, figsize=(16, 6), constrained_layout=True)
boundary = gpd.GeoSeries([study], crs=CRS_M)

selected_part.plot(ax=axes[0], color="#d6e8ef", edgecolor="none")
boundary.boundary.plot(ax=axes[0], color="#18394a", linewidth=0.8)
axes[0].set_title(f"Padova | selected H3 resolution {selected}")

focus = study.representative_point()
radius = 450
window = box(focus.x - radius, focus.y - radius, focus.x + radius, focus.y + radius)

for ax, res in zip(axes[1:], RESOLUTIONS):
    layer = grids[res][0]
    local = layer[layer.geometry.intersects(window)]
    local.boundary.plot(ax=ax, linewidth=0.8, color="#31708a")
    ax.set_xlim(focus.x - radius, focus.x + radius)
    ax.set_ylim(focus.y - radius, focus.y + radius)
    ax.set_title(f"H3 resolution {res} | 900 m x 900 m")

for ax in axes:
    ax.set_aspect("equal")
    ax.set_axis_off()

fig.savefig(OUT_RES / "h3_resolution_comparison.png", dpi=220, bbox_inches="tight")
plt.show()

## Extract 2022 reference-date features

In [ ]:
DATE = "2022-06-17"
HEX_FILE = OUT_RES / "h3_r10_study_part.gpkg"
RASTER_DIR = PROJECT_ROOT / "rasters" / "20220617" / "align"
OUT_2022 = BASE / "h3_r10_20220617_public_v1"
OUT_2022.mkdir(parents=True, exist_ok=True)

FILES = {
    "lst": "align_lst.tif",
    "ndvi": "align_ndvi.tif",
    "ndbi": "align_ndbi.tif",
    "ndwi": "align_ndwi.tif",
    "albedo": "align_albedo.tif",
    "bcr": "align_BCR.tif",
    "tcd": "align_TCD.tif",
}
FEATURES = ["ndvi", "ndbi", "ndwi", "albedo", "bcr", "tcd"]
VARIABLES = ["lst"] + FEATURES
MIN_COVERAGE = 0.80
RANGES = {
    "ndvi": (-1, 1), "ndbi": (-1, 1), "ndwi": (-1, 1),
    "albedo": (0, 1), "bcr": (0, 1), "tcd": (0, 100),
}

hexes = gpd.read_file(HEX_FILE, layer="study_part").sort_values("h3_id").reset_index(drop=True)

# Read reference raster grid info
with rasterio.open(RASTER_DIR / FILES["lst"]) as ref:
    ref_crs = ref.crs
    transform = ref.transform
    height, width = ref.height, ref.width

# Load and clean 2022 rasters
arrays = {}
checks = []
for name, filename in FILES.items():
    with rasterio.open(RASTER_DIR / filename) as src:
        if src.crs != ref_crs or src.shape != (height, width) or not src.transform.almost_equals(transform):
            raise ValueError(f"Raster misaligned: {filename}")
        raw = src.read(1, masked=True).astype("float64").filled(np.nan)
        values = raw * src.scales[0] + src.offsets[0]

    if name == "lst":
        valid = np.isfinite(values) & (values > 0) & (values < 100)
    else:
        lo, hi = RANGES[name]
        valid = np.isfinite(values) & (values >= lo) & (values <= hi)

    values[~valid] = np.nan
    arrays[name] = values
    checks.append({
        "variable": name,
        "valid_pixels": int(valid.sum()),
        "min": float(np.nanmin(values)),
        "max": float(np.nanmax(values)),
    })

checks_df = pd.DataFrame(checks)
display(checks_df.round(4))

# Compute intersection-area weighted means for each hexagon
means = np.full((len(hexes), len(VARIABLES)), np.nan)
coverages = np.zeros_like(means)
raster_coverage = np.zeros(len(hexes))

pw, ph = transform.a, -transform.e
west, north = transform.c, transform.f

for i, geom in enumerate(hexes.geometry):
    xmin, ymin, xmax, ymax = geom.bounds
    c0 = max(0, int(np.floor((xmin - west) / pw)))
    c1 = min(width, int(np.ceil((xmax - west) / pw)))
    r0 = max(0, int(np.floor((north - ymax) / ph)))
    r1 = min(height, int(np.ceil((north - ymin) / ph)))
    if r0 >= r1 or c0 >= c1:
        continue

    rr, cc = np.mgrid[r0:r1, c0:c1]
    rr, cc = rr.ravel(), cc.ravel()
    left = west + cc * pw
    top = north - rr * ph
    pixels = shapely.box(left, top - ph, left + pw, top)

    weights = shapely.area(shapely.intersection(pixels, geom))
    keep = weights > 0
    rr, cc, weights = rr[keep], cc[keep], weights[keep]

    area = geom.area
    raster_coverage[i] = weights.sum() / area

    for j, name in enumerate(VARIABLES):
        vals = arrays[name][rr, cc]
        ok = np.isfinite(vals)
        valid_area = weights[ok].sum()
        coverages[i, j] = valid_area / area
        if valid_area > 0:
            means[i, j] = np.dot(vals[ok], weights[ok]) / valid_area

# Assemble candidate and valid node tables
id_cols = ["h3_id", "resolution", "center_x", "center_y", "full_area_m2", "inside_area_m2", "inside_fraction"]
table = hexes[id_cols].copy()
table.insert(1, "date", DATE)
table["raster_coverage"] = np.clip(raster_coverage, 0, 1)

for j, name in enumerate(VARIABLES):
    table[name] = means[:, j]
    table[f"{name}_coverage"] = np.clip(coverages[:, j], 0, 1)

table["target_ok"] = table["lst"].notna() & (table["lst_coverage"] >= MIN_COVERAGE)
table["inputs_ok"] = table[FEATURES].notna().all(axis=1) & (table[[f"{v}_coverage" for v in FEATURES]] >= MIN_COVERAGE).all(axis=1)
table["sample_ok"] = table["target_ok"] & table["inputs_ok"]

nodes = table[table["sample_ok"]].copy().reset_index(drop=True)
nodes.insert(0, "local_node_id", np.arange(len(nodes), dtype=np.int64))

quality = pd.DataFrame({
    "variable": VARIABLES,
    "below_80pct": [int((table[f"{v}_coverage"] < MIN_COVERAGE).sum()) for v in VARIABLES],
    "zero_coverage": [int((table[f"{v}_coverage"] == 0).sum()) for v in VARIABLES],
})

table.to_csv(OUT_2022 / "h3_candidate_table.csv", index=False)
nodes.to_csv(OUT_2022 / "h3_node_table.csv", index=False)
quality.to_csv(OUT_2022 / "coverage_by_variable.csv", index=False)
checks_df.to_csv(OUT_2022 / "raster_checks.csv", index=False)

print(f"2022 valid hexagons: {len(nodes)} / {len(table)}")
display(quality)

## Build 6-direction H3 graph and 2022 training package

In [ ]:
GRAPH_OUT = OUT_2022 / "graph_directional"
GRAPH_OUT.mkdir(parents=True, exist_ok=True)

PACKAGE_2022 = BASE / "h3_r10_20220617_public_training_v1"
(PACKAGE_2022 / "splits").mkdir(parents=True, exist_ok=True)

DIRECTIONS = [f"D{k}" for k in range(6)]
SPLIT_SEEDS = [42, 123, 2026, 3407, 777]
INIT_SEEDS = [11, 22, 33]

nodes = pd.read_csv(OUT_2022 / "h3_node_table.csv", dtype={"h3_id": str}).sort_values("local_node_id").reset_index(drop=True)
ids = nodes["h3_id"].tolist()
n = len(ids)
id_to_local = {cell: i for i, cell in enumerate(ids)}

# Find theoretical 6 neighbors on the H3 grid
rings = {}
all_ids = set(ids)
for cell in ids:
    nbs = sorted(set(h3.grid_disk(cell, 1)) - {cell})
    rings[cell] = nbs
    all_ids.update(nbs)

# Project all cell centers to EPSG:32632 and compute bearings
all_ids = sorted(all_ids)
latlon = np.array([h3.cell_to_latlng(c) for c in all_ids])
proj = Transformer.from_crs("EPSG:4326", "EPSG:32632", always_xy=True)
xs, ys = proj.transform(latlon[:, 1], latlon[:, 0])
xy = dict(zip(all_ids, zip(xs, ys)))

angles = np.zeros((n, 6))
for i, cell in enumerate(ids):
    cx, cy = xy[cell]
    for k, nb_cell in enumerate(rings[cell]):
        nx, ny = xy[nb_cell]
        angles[i, k] = np.degrees(np.arctan2(nx - cx, ny - cy)) % 360

# Estimate common 6-fold orientation and assign each neighbor to D0..D5
phase = np.exp(1j * np.deg2rad(6 * angles)).mean()
d0 = np.rad2deg(np.angle(phase)) / 6
axes = (d0 + 60 * np.arange(6)) % 360
slots = np.floor(((angles - d0 + 30) % 360) / 60).astype(int)

if not np.all(np.sort(slots, axis=1) == np.arange(6)):
    raise ValueError("Two neighbors fell into the same bearing sector.")

# Build neighbor index matrix (-1 if neighbor is outside valid nodes)
neighbor_index = np.full((n, 6), -1, dtype=np.int64)
for i, cell in enumerate(ids):
    for nb_cell, k in zip(rings[cell], slots[i]):
        neighbor_index[i, k] = id_to_local.get(nb_cell, -1)

target, edge_type = np.where(neighbor_index >= 0)
source = neighbor_index[target, edge_type]
edge_index = np.stack([source, target]).astype(np.int64)
edge_type = edge_type.astype(np.int64)
degree = (neighbor_index >= 0).sum(axis=1)

# Save directional graph files
nb_table = nodes[["local_node_id", "h3_id", "center_x", "center_y"]].copy()
for k, d in enumerate(DIRECTIONS):
    nb_table[d] = [ids[j] if j >= 0 else pd.NA for j in neighbor_index[:, k]]
nb_table["n_neighbors"] = degree

dir_summary = pd.DataFrame({
    "edge_type": np.arange(6),
    "direction": DIRECTIONS,
    "bearing_degrees_clockwise_from_grid_north": axes,
    "opposite_direction": [DIRECTIONS[(k + 3) % 6] for k in range(6)],
    "valid_neighbors": np.bincount(edge_type, minlength=6),
})

nb_table.to_csv(GRAPH_OUT / "h3_ordered_neighbors.csv", index=False)
dir_summary.to_csv(GRAPH_OUT / "direction_definition.csv", index=False)
np.save(GRAPH_OUT / "neighbor_index.npy", neighbor_index)
np.save(GRAPH_OUT / "edge_index.npy", edge_index)
np.save(GRAPH_OUT / "edge_type.npy", edge_type)
display(dir_summary.round(2))

# Helper to build [center, neighbor_blocks, presence_flags]
def build_ordered_features(X_center, nb):
    present = nb >= 0
    blocks = np.full((*nb.shape, X_center.shape[1]), np.nan, dtype=np.float32)
    blocks[present] = X_center[nb[present]]
    return np.concatenate([X_center, blocks.reshape(len(X_center), -1), present.astype(np.float32)], axis=1)

X_center = nodes[FEATURES].to_numpy(dtype=np.float32)
y = nodes["lst"].to_numpy(dtype=np.float32)
X_ordered = build_ordered_features(X_center, neighbor_index)

ordered_columns = (
    [f"center_{f}" for f in FEATURES]
    + [f"{d}_{f}" for d in DIRECTIONS for f in FEATURES]
    + [f"{d}_exists" for d in DIRECTIONS]
)

# Generate 5 reproducible 80/10/10 splits (test first, val second, train remainder)
n_val = round(0.10 * n)
n_test = round(0.10 * n)
split_rows = []

for seed in SPLIT_SEEDS:
    order = np.random.default_rng(seed).permutation(n)
    test_mask = np.zeros(n, dtype=bool)
    val_mask = np.zeros(n, dtype=bool)
    train_mask = np.zeros(n, dtype=bool)

    test_mask[order[:n_test]] = True
    val_mask[order[n_test:n_test + n_val]] = True
    train_mask[order[n_test + n_val:]] = True
    final_train_mask = train_mask | val_mask

    np.savez_compressed(
        PACKAGE_2022 / "splits" / f"split_{seed}.npz",
        train_mask=train_mask, val_mask=val_mask, test_mask=test_mask, final_train_mask=final_train_mask,
    )
    split_rows.append({
        "split_seed": seed, "nodes": n,
        "train": int(train_mask.sum()), "val": int(val_mask.sum()),
        "test": int(test_mask.sum()), "final_train": int(final_train_mask.sum()),
    })

split_summary = pd.DataFrame(split_rows)
split_summary.to_csv(PACKAGE_2022 / "split_summary.csv", index=False)

# Save 2022 package arrays and config
np.save(PACKAGE_2022 / "X_center_raw.npy", X_center)
np.save(PACKAGE_2022 / "X_ordered_raw.npy", X_ordered)
np.save(PACKAGE_2022 / "y_lst_C.npy", y)
np.save(PACKAGE_2022 / "neighbor_index.npy", neighbor_index)
np.save(PACKAGE_2022 / "edge_index.npy", edge_index)
np.save(PACKAGE_2022 / "edge_type.npy", edge_type)

nodes[["local_node_id", "h3_id", "center_x", "center_y"]].to_csv(PACKAGE_2022 / "node_order.csv", index=False)
shutil.copy2(GRAPH_OUT / "direction_definition.csv", PACKAGE_2022)
shutil.copy2(GRAPH_OUT / "h3_ordered_neighbors.csv", PACKAGE_2022)

config_2022 = {
    "date": "2022-06-17",
    "spatial_representation": "H3",
    "h3_resolution": 10,
    "feature_group": "public",
    "features": FEATURES,
    "target": "central cell LST in degrees Celsius",
    "ordered_columns": ordered_columns,
    "directions": DIRECTIONS,
    "edge_convention": "source=neighbor, target=center",
    "n_nodes": n,
    "n_undirected_edges": int(edge_index.shape[1] // 2),
    "split_seeds": SPLIT_SEEDS,
    "initialization_seeds": INIT_SEEDS,
    "arrays_are_standardized": False,
}
with open(PACKAGE_2022 / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(config_2022, f, indent=2)

shutil.make_archive(str(PACKAGE_2022) + "_backup", "zip", root_dir=PACKAGE_2022.parent, base_dir=PACKAGE_2022.name)
display(split_summary)

## Align 2023 & 2024 rasters and find three-date common nodes

In [ ]:
EXTRA_DATES = ["2023-08-15", "2024-08-25"]
PREP_OUT = BASE / "h3_r10_three_dates_public_preparation_v1"
PREP_OUT.mkdir(parents=True, exist_ok=True)

hexes = gpd.read_file(HEX_FILE, layer="study_part").sort_values("h3_id").reset_index(drop=True)
first = pd.read_csv(OUT_2022 / "h3_node_table.csv", dtype={"h3_id": str}).sort_values("local_node_id").reset_index(drop=True)

with rasterio.open(PROJECT_ROOT / "rasters" / "20220617" / "align" / FILES["lst"]) as ref:
    transform, shape, crs = ref.transform, ref.shape, ref.crs

ranges_all = {"lst": (0, 100), **RANGES}
arrays = {}
lst_zero_masks = {}
checks = []

# Load and clean 2023 and 2024 rasters
for date in EXTRA_DATES:
    folder = PROJECT_ROOT / "rasters" / date.replace("-", "") / "align"
    arrays[date] = {}
    for var in VARIABLES:
        path = folder / FILES[var]
        with rasterio.open(path) as src:
            if src.crs != crs or src.shape != shape or not src.transform.almost_equals(transform):
                raise ValueError(f"Raster misaligned: {path}")
            raw = src.read(1, masked=True).astype("float64").filled(np.nan)
            values = raw * src.scales[0] + src.offsets[0]

        lo, hi = ranges_all[var]
        valid = np.isfinite(values) & (values >= lo) & (values <= hi)
        count = int(valid.sum())
        values[~valid] = np.nan

        # Track zero-valued LST pixels and mask them out
        if var == "lst":
            lst_zero_masks[date] = valid & (values == 0)
            values[lst_zero_masks[date]] = np.nan

        arrays[date][var] = values
        checks.append({"date": date, "variable": var, "valid_pixels": count})

checks_extra = pd.DataFrame(checks)
display(checks_extra)

# Compute area-weighted means and coverage for 2023 and 2024
height, width = shape
pw, ph = transform.a, -transform.e
west, north = transform.c, transform.f
n_cand = len(hexes)

means = {d: np.full((n_cand, len(VARIABLES)), np.nan) for d in EXTRA_DATES}
coverage = {d: np.zeros((n_cand, len(VARIABLES))) for d in EXTRA_DATES}
zero_area = {d: np.zeros(n_cand) for d in EXTRA_DATES}
raster_fraction = np.zeros(n_cand)

for i, geom in enumerate(hexes.geometry):
    xmin, ymin, xmax, ymax = geom.bounds
    c0 = max(0, int(np.floor((xmin - west) / pw)))
    c1 = min(width, int(np.ceil((xmax - west) / pw)))
    r0 = max(0, int(np.floor((north - ymax) / ph)))
    r1 = min(height, int(np.ceil((north - ymin) / ph)))
    if r0 >= r1 or c0 >= c1:
        continue

    rr, cc = np.mgrid[r0:r1, c0:c1]
    rr, cc = rr.ravel(), cc.ravel()
    left = west + cc * pw
    top = north - rr * ph
    pixels = shapely.box(left, top - ph, left + pw, top)

    weights = shapely.area(shapely.intersection(pixels, geom))
    keep = weights > 0
    rr, cc, weights = rr[keep], cc[keep], weights[keep]
    area = geom.area
    raster_fraction[i] = weights.sum() / area

    for d in EXTRA_DATES:
        zero_area[d][i] = weights[lst_zero_masks[d][rr, cc]].sum()
        for j, var in enumerate(VARIABLES):
            vals = arrays[d][var][rr, cc]
            ok = np.isfinite(vals)
            valid_area = weights[ok].sum()
            coverage[d][i, j] = valid_area / area
            if valid_area > 0:
                means[d][i, j] = np.dot(weights[ok], vals[ok]) / valid_area

# Filter valid cells and find three-date common nodes
first_ids = set(first["h3_id"])
valid_ids = {}
report = []
zero_reports = []

for d in EXTRA_DATES:
    t = hexes[id_cols].copy()
    t["date"] = d
    t["raster_coverage"] = np.clip(raster_fraction, 0, 1)
    t["lst_zero_overlap_m2"] = zero_area[d]

    for j, var in enumerate(VARIABLES):
        t[var] = means[d][:, j]
        t[f"{var}_coverage"] = np.clip(coverage[d][:, j], 0, 1)

    t["target_ok"] = t["lst"].notna() & (t["lst_coverage"] >= MIN_COVERAGE)
    t["inputs_ok"] = t[FEATURES].notna().all(axis=1) & (t[[f"{v}_coverage" for v in FEATURES]] >= MIN_COVERAGE).all(axis=1)
    t["sample_ok"] = t["target_ok"] & t["inputs_ok"]
    t["in_2022_experiment"] = t["h3_id"].isin(first_ids)
    t.to_csv(PREP_OUT / f"h3_candidates_{d}.csv", index=False)

    valid_ids[d] = set(t.loc[t["sample_ok"], "h3_id"])
    affected = t.loc[t["lst_zero_overlap_m2"] > 0, ["date", "h3_id", "in_2022_experiment", "lst_zero_overlap_m2", "lst_coverage", "sample_ok"]]
    zero_reports.append(affected)

    report.append({
        "date": d,
        "all_candidates": len(t),
        "valid_on_date": int(t["sample_ok"].sum()),
        "valid_among_2022_nodes": len(first_ids & valid_ids[d]),
        "lost_from_2022_nodes": len(first_ids - valid_ids[d]),
        "2022_nodes_touching_LST_zero": int(affected["in_2022_experiment"].sum()),
    })

common_ids = first_ids & valid_ids[EXTRA_DATES[0]] & valid_ids[EXTRA_DATES[1]]
common_order = first.loc[first["h3_id"].isin(common_ids), ["local_node_id", "h3_id"]].rename(
    columns={"local_node_id": "original_2022_node_id"}
).copy()
common_order.insert(0, "common_node_id", np.arange(len(common_order)))

report_df = pd.DataFrame(report)
common_order.to_csv(PREP_OUT / "h3_common_node_order.csv", index=False)
report_df.to_csv(PREP_OUT / "date_quality_summary.csv", index=False)
pd.concat(zero_reports, ignore_index=True).to_csv(PREP_OUT / "lst_zero_overlap.csv", index=False)
checks_extra.to_csv(PREP_OUT / "raster_checks.csv", index=False)

print("Common H3 nodes across all 3 dates:", len(common_order))
display(report_df)

## Export the three-date common subgraph and training arrays

In [ ]:
OUT_COMMON = BASE / "h3_r10_three_dates_public_common5712_v1"
SHARED = OUT_COMMON / "shared"
SPLITS_DIR = SHARED / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

ALL_DATES = ["2022-06-17", "2023-08-15", "2024-08-25"]

# Map 2022 node indices to common node indices
old_order = pd.read_csv(PACKAGE_2022 / "node_order.csv", dtype={"h3_id": str}).sort_values("local_node_id").reset_index(drop=True)
common = pd.read_csv(PREP_OUT / "h3_common_node_order.csv", dtype={"h3_id": str}).sort_values("common_node_id").reset_index(drop=True)

n_old, n_common = len(old_order), len(common)
old_idx = common["original_2022_node_id"].to_numpy(dtype=np.int64)
old_to_new = np.full(n_old, -1, dtype=np.int64)
old_to_new[old_idx] = np.arange(n_common)

order = old_order.iloc[old_idx][["h3_id", "center_x", "center_y"]].copy().reset_index(drop=True)
order.insert(0, "local_node_id", np.arange(n_common))
order.insert(1, "original_2022_node_id", old_idx)
order.to_csv(SHARED / "node_order.csv", index=False)

# Save LST zero-overlap report for original 2022 nodes
zero = pd.read_csv(PREP_OUT / "lst_zero_overlap.csv", dtype={"h3_id": str})
zero = zero[zero["h3_id"].isin(old_order["h3_id"])].copy()
zero["retained_all_three_dates"] = zero["h3_id"].isin(common["h3_id"])
zero.to_csv(SHARED / "lst_zero_overlap_original_nodes.csv", index=False)

# Restrict directional graph to common nodes while keeping D0..D5
old_nb = np.load(PACKAGE_2022 / "neighbor_index.npy")
selected_old_nb = old_nb[old_idx]
nb = np.full((n_common, 6), -1, dtype=np.int64)
has_old = selected_old_nb >= 0
nb[has_old] = old_to_new[selected_old_nb[has_old]]

present = nb >= 0
dst, edge_type = np.where(present)
src = nb[dst, edge_type]
edge_index = np.stack([src, dst]).astype(np.int64)
edge_type = edge_type.astype(np.int64)

np.save(SHARED / "neighbor_index.npy", nb)
np.save(SHARED / "edge_index.npy", edge_index)
np.save(SHARED / "edge_type.npy", edge_type)

ids_common = order["h3_id"].to_numpy()
nb_table = order.copy()
for k, d in enumerate(DIRECTIONS):
    nb_table[d] = [ids_common[j] if j >= 0 else pd.NA for j in nb[:, k]]
nb_table["n_neighbors"] = present.sum(axis=1)
nb_table.to_csv(SHARED / "h3_ordered_neighbors.csv", index=False)

dir_table = pd.read_csv(PACKAGE_2022 / "direction_definition.csv").sort_values("edge_type").reset_index(drop=True)
dir_table["valid_neighbors"] = np.bincount(edge_type, minlength=6)
dir_table.to_csv(SHARED / "direction_definition.csv", index=False)

# Restrict 2022 split masks to common nodes
split_rows = []
for seed in SPLIT_SEEDS:
    with np.load(PACKAGE_2022 / "splits" / f"split_{seed}.npz") as f:
        masks = {k: f[k][old_idx].astype(bool) for k in ["train_mask", "val_mask", "test_mask", "final_train_mask"]}

    tr, va, te = masks["train_mask"], masks["val_mask"], masks["test_mask"]
    assert np.all(tr.astype(int) + va.astype(int) + te.astype(int) == 1), f"Invalid split masks for seed {seed}"

    np.savez_compressed(SPLITS_DIR / f"split_{seed}.npz", **masks)
    split_rows.append({
        "split_seed": seed, "nodes": n_common,
        "train": int(tr.sum()), "val": int(va.sum()),
        "test": int(te.sum()), "final_train": int((tr | va).sum()),
    })

split_summary_common = pd.DataFrame(split_rows)
split_summary_common.to_csv(SHARED / "split_summary.csv", index=False)

# Export aligned feature arrays for all three dates
source_tables = {
    "2022-06-17": OUT_2022 / "h3_node_table.csv",
    "2023-08-15": PREP_OUT / "h3_candidates_2023-08-15.csv",
    "2024-08-25": PREP_OUT / "h3_candidates_2024-08-25.csv",
}

date_rows = []
for date in ALL_DATES:
    t = pd.read_csv(source_tables[date], dtype={"h3_id": str}).set_index("h3_id").loc[ids_common]
    X = t[FEATURES].to_numpy(dtype=np.float32)
    y = t["lst"].to_numpy(dtype=np.float32)
    X_ordered = build_ordered_features(X, nb)

    date_out = OUT_COMMON / date
    date_out.mkdir(exist_ok=True)
    np.save(date_out / "X_center_raw.npy", X)
    np.save(date_out / "X_ordered_raw.npy", X_ordered)
    np.save(date_out / "y_lst_C.npy", y)

    date_rows.append({
        "date": date, "nodes": n_common,
        "features": X.shape[1], "ordered_features": X_ordered.shape[1],
        "LST_mean_C": float(y.mean()), "LST_std_C": float(y.std()),
        "LST_min_C": float(y.min()), "LST_max_C": float(y.max()),
    })

date_summary = pd.DataFrame(date_rows)
date_summary.to_csv(OUT_COMMON / "date_summary.csv", index=False)

config_common = {
    "dates": ALL_DATES,
    "feature_group": "public",
    "features": FEATURES,
    "h3_resolution": 10,
    "common_nodes": n_common,
    "undirected_edges": int(edge_index.shape[1] // 2),
    "directed_entries": int(edge_index.shape[1]),
    "directions": DIRECTIONS,
    "split_seeds": SPLIT_SEEDS,
    "init_seeds": INIT_SEEDS,
    "split_method": "Original 2022 H3 masks restricted to common H3 IDs",
    "edge_convention": "source = neighbor; destination = center",
    "arrays_are_standardized": False,
}
with open(OUT_COMMON / "experiment_config.json", "w", encoding="utf-8") as f:
    json.dump(config_common, f, indent=2)

shutil.make_archive(str(OUT_COMMON) + "_backup", "zip", root_dir=OUT_COMMON.parent, base_dir=OUT_COMMON.name)

display(split_summary_common)
display(date_summary.round(4))